# KBO 크롤러 직접 실행 노트북

이 노트북을 위에서 아래로 실행하면 schedule → result → relay → community 순으로
KBO 원본 데이터를 S3(브론즈 랜딩존)에 적재합니다.

**사전 준비**
1. `cd py-collector && pip install -e ".[dev]"`
2. `.env` 설정 (실 AWS면 `COLLECTOR_S3_ENDPOINT` 비움 + `aws configure` 완료 / LocalStack이면 `http://localhost:4566`)
3. **`py-collector/` 디렉토리에서** `jupyter lab` 실행 (그래야 `.env`와 `config/targets.yaml`을 찾습니다)

> 적재 키는 멱등이라 여러 번 돌려도 같은 데이터는 덮어써집니다(중복 없음). 이미 있는 건 건너뜁니다.

## 0. 설정 로드 & 대상 확인

> 이 셀은 커널의 작업 디렉토리와 무관하게 **`py-collector/` 루트를 찾아 이동**한 뒤 `.env`를 로드합니다.
> (주피터/PyCharm이 노트북 폴더나 프로젝트 루트를 CWD로 잡아도 동작하도록.)

In [4]:
import os, json, uuid, datetime, collections
from pathlib import Path

# --- py-collector 루트로 이동 (.env / config 를 찾기 위해; 커널 CWD 무관) ---
def _find_root():
    # 1) 현재 위치에서 위로
    for c in [Path.cwd(), *Path.cwd().parents]:
        if (c / "kbo_collector").is_dir() and (c / "config" / "targets.yaml").exists():
            return c
    # 2) 아래로 (repo 루트/상위에서 연 경우)
    for pat in ("*/kbo_collector", "*/*/kbo_collector", "*/*/*/kbo_collector"):
        for m in sorted(Path.cwd().glob(pat)):
            root = m.parent
            if (root / "config" / "targets.yaml").exists():
                return root
    return None

_root = _find_root()
assert _root, f"py-collector 루트를 못 찾음 (현재 CWD: {Path.cwd()}). py-collector 안에서 주피터를 실행하세요."
os.chdir(_root)
assert Path(".env").exists(), (
    f".env 가 {Path.cwd()} 에 없습니다. `cp .env.example .env` 후 버킷/PII salt/AWS 값을 채우세요.")
print("작업 디렉토리:", Path.cwd())

from kbo_collector.config import get_settings
from kbo_collector import fetch, naver, run
from kbo_collector.journal import Journal, setup_logging
from kbo_collector.sink import S3RawSink

setup_logging()
get_settings.cache_clear()          # 이전(실패) 캐시 제거 후 재로드
settings = get_settings()
sink = S3RawSink(settings)
client = fetch.build_client(settings)

print("적재 대상 S3")
print("  버킷      :", settings.s3_bucket)
print("  엔드포인트:", settings.s3_endpoint or "실제 AWS S3")
print("  리전      :", settings.s3_region)
print("→ 이 대상이 맞는지 확인하세요. 바꾸려면 .env 를 수정하고 커널 재시작.")

작업 디렉토리: /Users/sotaeho/PycharmProjects/VictoryFairy/VitoryFairy_BE/py-collector
적재 대상 S3
  버킷      : victoryfairy-crawl-local
  엔드포인트: 실제 AWS S3
  리전      : ap-northeast-2
→ 이 대상이 맞는지 확인하세요. 바꾸려면 .env 를 수정하고 커널 재시작.


## 1. 수집 날짜 선택

`result`/`relay`를 보려면 **경기가 있는 날짜**를 고르세요. 경기 없는 날이면 schedule만 적재되고
gameId가 0개라 result/relay는 비어 있습니다(정상).

In [5]:
DATE = "2026-07-08"          # 예: 지난 경기일. 오늘이 경기 없는 날이면 다른 날짜로.
run_id = uuid.uuid4().hex[:8]
print("수집 날짜:", DATE, "| run_id:", run_id)

수집 날짜: 2026-07-08 | run_id: 4c9243e1


## 2. schedule 적재 → KBO 경기 ID 추출

In [6]:
game_ids = run.land_schedule(
    DATE, settings=settings, sink=sink, client=client,
    journal=Journal("schedule", DATE, run_id, settings.journal_dir))
print(f"KBO 정규 경기 {len(game_ids)}개")
game_ids

KBO 정규 경기 5개


['20260708NCHH02026',
 '20260708HTLT02026',
 '20260708LGSS02026',
 '20260708SKOB02026',
 '20260708WOKT02026']

### 경기 상태 살펴보기 (읽기 전용)

In [7]:
def classify(g):
    if g.get("cancel"):    return "취소"
    if g.get("suspended"): return "서스펜디드"
    return {"RESULT": "완료", "BEFORE": "예정", "LIVE": "진행중"}.get(g.get("statusCode"), g.get("statusCode"))

r = fetch.fetch(client, naver.schedule_url(settings, DATE), settings=settings, referer=settings.naver_referer)
games = [g for g in (r.json().get("result", {}).get("games") or []) if g.get("categoryId") == "kbo"]
if not games:
    print(DATE, "→ 일정 없음 (KBO 경기 없음)")
for g in games:
    print(f"  {g['awayTeamName']:>4} @ {g['homeTeamName']:<4}  {classify(g):<6} "
          f"({g.get('statusInfo') or '-'})  {g['gameId']}")

    NC @ 한화    취소     (경기취소)  20260708NCHH02026
   KIA @ 롯데    완료     (9회초)  20260708HTLT02026
    LG @ 삼성    완료     (9회말)  20260708LGSS02026
   SSG @ 두산    완료     (9회초)  20260708SKOB02026
    키움 @ KT    완료     (9회초)  20260708WOKT02026


## 3. result 적재 (경기별 결과 JSON)

In [8]:
n = run.land_results(
    DATE, game_ids, settings=settings, sink=sink, client=client,
    journal=Journal("result", DATE, run_id, settings.journal_dir))
print(f"result 적재: {n}개")

result 적재: 0개


## 4. relay 적재 (경기별 이닝 문자중계)

취소 경기와 경기 범위 밖 이닝은 자동으로 건너뜁니다.

In [9]:
n = run.land_relays(
    DATE, game_ids, settings=settings, sink=sink, client=client,
    journal=Journal("relay", DATE, run_id, settings.journal_dir))
print(f"relay 이닝 적재: {n}개")

relay 이닝 적재: 0개


## 5. community 적재 (커뮤니티 글)

기본은 **빠른 테스트용으로 DCInside 갤러리 1개**만 돕니다. 전체 11개 대상(DC 10 + FMKorea)은
글당 0.8초 지연 때문에 수 분 걸리므로 아래 주석을 참고하세요.

> 커뮤니티 글은 게임 날짜가 아니라 '지금 인기글'을 수집합니다. `DATE`는 저장 파티션에만 쓰입니다.
> FMKorea는 Cloudflare 차단으로 0건일 수 있습니다.

In [11]:
import tempfile, pathlib

# 빠른 테스트: DCInside 두산 갤러리 1개만
quick = pathlib.Path(tempfile.gettempdir()) / "quick_targets.yaml"
quick.write_text(
    'targets:\n  - { source: DCINSIDE, url: "https://gall.dcinside.com/board/lists/?id=doosanbears_new1", team: DOOSAN }\n',
    encoding="utf-8")
settings.targets_file = str(quick)

n = run.land_community(
    DATE, settings=settings, sink=sink, client=client,
    journal=Journal("community", DATE, run_id, settings.journal_dir))
print(f"community 적재: {n}개")

# --- 전체 대상(config/targets.yaml)으로 돌리려면 아래 주석을 해제 (수 분 소요) ---
# settings.targets_file = "config/targets.yaml"
# n = run.land_community(DATE, settings=settings, sink=sink, client=client,
#                        journal=Journal("community", DATE, run_id, settings.journal_dir))
# print("전체 community 적재:", n)

community 적재: 0개


## 6. 적재 결과 확인 (S3)

In [ ]:
def peek(prefix):
    resp = sink.client.list_objects_v2(Bucket=settings.s3_bucket, Prefix=prefix)
    keys = [o["Key"] for o in resp.get("Contents", [])]
    return resp.get("KeyCount", 0), keys[:3]

for label, prefix in [
    ("schedule",  f"raw-json/schedule/{DATE}/"),
    ("result",    f"raw-json/result/{DATE}/"),
    ("relay",     "raw-json/relay/"),
    ("community", f"community/dcinside/{DATE}/"),
]:
    c, sample = peek(prefix)
    print(f"{label:<10} {c:>4}개   예) {sample}")

### 샘플 하나 열어보기

In [ ]:
if game_ids:
    key = f"raw-json/result/{DATE}/{game_ids[0]}.json"
    g = json.loads(sink.client.get_object(Bucket=settings.s3_bucket, Key=key)["Body"].read())["result"]["game"]
    print(f"{g['awayTeamName']} {g['awayTeamScore']} : {g['homeTeamScore']} {g['homeTeamName']}"
          f"  | {g['statusInfo']} | {g['stadium']} | 승: {g.get('winPitcherName')}")
else:
    print("game_ids가 비어 있어요(경기 없는 날). DATE를 경기 있는 날짜로 바꿔보세요.")

## (옵션) 7월 전체 일정 한눈에 보기

각 날짜의 경기 상태(완료/취소/예정/일정없음)를 표로 확인합니다. 적재는 하지 않고 조회만 합니다.

In [12]:
def classify(g):
    if g.get("cancel"):    return "취소"
    if g.get("suspended"): return "서스펜디드"
    return {"RESULT": "완료", "BEFORE": "예정", "LIVE": "진행중"}.get(g.get("statusCode"), g.get("statusCode"))

for d in range(1, 32):
    date = f"2026-07-{d:02d}"
    rr = fetch.fetch(client, naver.schedule_url(settings, date), settings=settings, referer=settings.naver_referer)
    gs = [g for g in (rr.json().get("result", {}).get("games") or []) if g.get("categoryId") == "kbo"]
    wd = "월화수목금토일"[datetime.date.fromisoformat(date).weekday()]
    if not gs:
        print(f"{date} ({wd})  일정 없음")
        continue
    c = collections.Counter(classify(g) for g in gs)
    print(f"{date} ({wd})  " + " | ".join(f"{k} {v}" for k, v in c.items()))

2026-07-01 (수)  완료 5
2026-07-02 (목)  완료 5
2026-07-03 (금)  완료 5
2026-07-04 (토)  완료 5
2026-07-05 (일)  완료 3 | 취소 2
2026-07-06 (월)  일정 없음
2026-07-07 (화)  완료 5
2026-07-08 (수)  취소 1 | 완료 4
2026-07-09 (목)  취소 1 | 완료 4
2026-07-10 (금)  일정 없음
2026-07-11 (토)  완료 1
2026-07-12 (일)  일정 없음
2026-07-13 (월)  일정 없음
2026-07-14 (화)  일정 없음
2026-07-15 (수)  일정 없음
2026-07-16 (목)  예정 5
2026-07-17 (금)  예정 5
2026-07-18 (토)  예정 5
2026-07-19 (일)  예정 5
2026-07-20 (월)  일정 없음
2026-07-21 (화)  예정 5
2026-07-22 (수)  예정 5
2026-07-23 (목)  예정 5
2026-07-24 (금)  예정 5
2026-07-25 (토)  예정 5
2026-07-26 (일)  예정 5
2026-07-27 (월)  일정 없음
2026-07-28 (화)  예정 5
2026-07-29 (수)  예정 5
2026-07-30 (목)  예정 5
2026-07-31 (금)  예정 5


## 정리

다 쓰면 HTTP 클라이언트를 닫습니다.

In [ ]:
client.close()
print("완료. 적재된 데이터는 위 5·6번 셀 또는 `aws s3 ls s3://%s/ --recursive` 로 확인하세요." % settings.s3_bucket)